In [ ]:
%cd "C:\Users\asmaa\Desktop\Amr\Master's Thesis v0.4"

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# !pip install scikit-optimize

In [ ]:
# from skopt import gp_minimize
# from skopt import space

In [ ]:
import devices
import models
import samplers
import trainers
import utils

import torch
import numpy as np
import matplotlib.pyplot as plt

from scipy.io import loadmat

In [ ]:
# parameters
lambd = 1

# wave number and normalized spacing
k_0 = 2 * np.pi / lambd
dx = (lambd / 40) * k_0

# normalized simulation lengths
substrate_length = (4 * lambd) * k_0
core_length = (1 * lambd) * k_0
cladding_length = (4 * lambd) * k_0

# refractive indices
n_substrate = 1
n_core = 3
n_cladding = 1

In [ ]:
study = 'TM'
field_type = 'E'

In [ ]:
RIs = [n_substrate, n_core, n_cladding]
lengths = [substrate_length, core_length, cladding_length]

In [ ]:
reference_device = devices.SlabWaveguide(RIs, lambd, lengths, study=study, field_type=field_type)

In [ ]:
memory_type = torch.device('cpu')
dtype = torch.float64
torch.manual_seed(42)
num_modes = 3

In [ ]:
base_model = models.discontinuity_capturing_network(1, 128, num_modes, 2, 'Siren')
optimzer = torch.optim.Adamax(base_model.parameters(), lr=1e-2)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimzer, milestones=[3000, 4000, 5000], gamma=0.2)
print('number of trainable parameters', sum([p.numel() for p in base_model.parameters() if p.requires_grad]))

In [ ]:
device = devices.SlabWaveguideWithPINNs(RIs, lambd, lengths, study, field_type, base_model, True)

In [ ]:
train_agent = trainers.Trainer(dtype, device, memory_type, [samplers.uniform_grid], optimzer, scheduler, dx)

In [ ]:
loss_history, rayleigh_history, mode_errors_list = train_agent.train(10001, num_modes, weights=[1/16, 1/2, 16, 1, 1, 1], verbose=1, sample_new=False)

# FD Performance

In [ ]:
print(dx)

In [ ]:
fd_device = devices.SlabWaveguideWithFD(RIs, lambd, lengths, study, field_type, dx)
_, x, u = fd_device.evaluate(num_modes)
utils.compare_against_analytic(fd_device, x, torch.from_numpy(u));

In [ ]:
save_folder_path = r"C:\Users\asmaa\Desktop\Amr\2nd Paper\Experiments"

In [ ]:
# fd_device = devices.SlabWaveguideWithFD(RIs, lambd, lengths, study, field_type, dx)
# 
# evaluation_points, eigen_modes_analytic, eigen_funcs_analytic = fd_device.evaluate_analytical(x, num_modes)
# 
# if fd_device.study == 'TM' and fd_device.field_type=='E':
#   print('adjusting analytic solution for Ex in a TM study')
#   eigen_funcs_analytic = eigen_funcs_analytic / fd_device.refractive_index(evaluation_points.reshape(-1), format='numpy')**2
# 
# eigen_funcs_analytic /= np.max(np.abs(eigen_funcs_analytic), axis=-1, keepdims=True)
# 
# print(eigen_funcs_analytic.shape, eigen_modes_analytic.shape)
# 
# np.save(save_folder_path + r"\Planar\Several Modes High Contrast\analytic_EIs.npy", eigen_modes_analytic)
# np.save(save_folder_path + r"\Planar\Several Modes High Contrast\analytic_fields.npy", eigen_funcs_analytic)

In [ ]:
# _, x, u = fd_device.evaluate(num_modes)
# print(_.shape, u.T.shape)
# u = u/np.max(np.abs(u), axis=0, keepdims=True)
# np.save(save_folder_path + r"\Planar\Several Modes High Contrast\FD_EIs.npy", _)
# np.save(save_folder_path + r"\Planar\Several Modes High Contrast\FD_fields.npy", u.T)

In [ ]:
# plt.plot(eigen_funcs_analytic[2])
# plt.plot(u.T[2], linestyle='--')

In [ ]:
mode_errors = []
num_modes = len(u.T)
evaluation_points, eigen_modes_analytic, eigen_funcs_analytic = reference_device.evaluate_analytical(x, num_modes)

if reference_device.study == 'TM':
  evaluation_pts_error = (torch.mean((torch.from_numpy(evaluation_points).reshape(-1) - x.reshape(-1)) ** 2) / np.mean(
          evaluation_points.reshape(-1) ** 2)) ** 0.5 * 100
  print(f'evalution points are at an error of {round(evaluation_pts_error.item(), 5)}%')

  if reference_device.field_type=='E':
    eigen_funcs_analytic = eigen_funcs_analytic / device.refractive_index(evaluation_points, format='numpy')**2
    eigen_funcs_analytic /= np.linalg.norm(eigen_funcs_analytic, axis=-1, keepdims=True)

# NN Performance

In [ ]:
augmented_x = torch.cat((torch.from_numpy(evaluation_points).reshape(-1, 1), device.refractive_index(torch.from_numpy(evaluation_points).reshape(-1, 1), format='torch')**2), dim=-1).to(memory_type)
predictions = device.underlying_model(augmented_x)
prediction_normed = predictions.detach().cpu() / torch.max(torch.abs(predictions.detach().cpu()), dim=0, keepdims=True)[0]

utils.compare_against_analytic(reference_device, x, prediction_normed.detach().cpu());

In [ ]:
x_tensor = torch.from_numpy(x)
x_tensor.requires_grad = True
x_augmented = torch.cat((x_tensor.reshape(-1, 1), device.refractive_index(torch.from_numpy(x).reshape(-1, 1), format='torch')**2), dim=-1)
NN_EI = [device.calculate_eigen_value(x_augmented.to(memory_type), 0).item()**(1/2), device.calculate_eigen_value(x_augmented.to(memory_type), 1).item()**(1/2), device.calculate_eigen_value(x_augmented.to(memory_type), 2).item()**(1/2)]

_**(1/2), '\n', reference_device.evaluate_analytical(x, 3)[1], '\n', NN_EI

In [ ]:
np.save(save_folder_path + r"\Planar\Several Modes High Contrast\DCNN_EIs.npy", NN_EI)
np.save(save_folder_path + r"\Planar\Several Modes High Contrast\DCNN_fields.npy", prediction_normed.T.numpy())

In [ ]:
test = np.load(save_folder_path + r"\Planar\Several Modes High Contrast\DCNN_EIs.npy")
print(test.shape)
test = np.load(save_folder_path + r"\Planar\Several Modes High Contrast\DCNN_fields.npy")
print(test.shape)
plt.plot(test[1]);
plt.plot()